In [1]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

/mnt/home/test/beluga-call-pipeline/


In [5]:

import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import librosa
from data_preprocessing.spectrogram.spectrogram_generator import HYDROPHONE_SENSITIVITY, SpectrogramGenerator
import pandas as pd
import torch

from tqdm import tqdm
import os
pd.set_option('display.max_columns', None)


In [ ]:
#Loading the labels for the overlaps
labels_df = pd.read_csv("../data/Verified_Dataset/labels/labels_merged.csv")


,clip_filename,clip_1,clip_2,clip_3,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,Site,HydrophoneModel,HydrophoneSensitivity,original_filename,labeled_snippet_filename,clip_start_time,clip_end_time
0,BSM_20170724_13385800_s3.wav,BSM_20170724_13385800.wav,BSM_20170724_13385900.wav,BSM_20170724_13390000.wav,1,0,0,0,1,1.0,evaluation_v1,BSM,201359382,-172.7,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58.000000000,2017-07-24 13:39:01.000000000
1,BSM_20170724_13385900_s3.wav,BSM_20170724_13385900.wav,BSM_20170724_13390000.wav,BSM_20170724_13390100.wav,1,0,0,0,1,1.0,evaluation_v1,BSM,201359382,-172.7,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:59.000000000,2017-07-24 13:39:02.000000000
2,BSM_20170724_13390000_s3.wav,BSM_20170724_13390000.wav,BSM_20170724_13390100.wav,BSM_20170724_13390200.wav,1,0,0,0,1,1.0,evaluation_v1,BSM,201359382,-172.7,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:39:00.000000000,2017-07-24 13:39:03.000000000
3,BSM_20170724_13390100_s3.wav,BSM_20170724_13390100.wav,BSM_20170724_13390200.wav,BSM_20170724_13390300.wav,1,0,0,0,1,1.0,evaluation_v1,BSM,201359382,-172.7,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:39:01.000000000,2017-07-24 13:39:04.000000000
4,BSM_20170724_13390200_s3.wav,BSM_20170724_13390200.wav,BSM_20170724_13390300.wav,BSM_20170724_13390400.wav,1,0,0,0,1,1.0,evaluation_v1,BSM,201359382,-172.7,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:39:02.000000000,2017-07-24 13:39:05.000000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9740,RDL_20200901_19191574_s3.wav,RDL_20200901_19191574.wav,RDL_20200901_19191674.wav,RDL_20200901_19191774.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,RDL,5675,-176.5,5675.200901190955.wav,NaN,2020-09-01 19:19:15.740000,2020-09-01 19:19:18.740000
9741,RDL_20200901_19192312_s3.wav,RDL_20200901_19192312.wav,RDL_20200901_19192412.wav,RDL_20200901_19192512.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,RDL,5675,-176.5,5675.200901190955.wav,NaN,2020-09-01 19:19:23.120000,2020-09-01 19:19:26.120000
9742,RDL_20200906_14553196_s3.wav,RDL_20200906_14553196.wav,RDL_20200906_14553296.wav,RDL_20200906_14553396.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,RDL,5675,-176.5,5675.200906144958.wav,NaN,2020-09-06 14:55:31.960000,2020-09-06 14:55:34.960000
9743,RDL_20200906_14555838_s3.wav,RDL_20200906_14555838.wav,RDL_20200906_14555938.wav,RDL_20200906_14560038.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,RDL,5675,-176.5,5675.200906144958.wav,NaN,2020-09-06 14:55:58.380000,2020-09-06 14:56:01.380000


In [17]:
import pandas as pd
import numpy as np
from datetime import timedelta

def create_3second_windows(labels_df):
    """
    Create 3-second windows from consecutive 1-second rows.
    
    Parameters:
    -----------
    labels_df : pd.DataFrame
        DataFrame with 1-second window rows
        
    Returns:
    --------
    pd.DataFrame
        New DataFrame with 3-second windows
    """
    # Make a copy
    df = labels_df.copy()
    
    # Convert clip_start_time to datetime if it's not already
    df['clip_start_time'] = pd.to_datetime(df['clip_start_time'], format='mixed')
    df['clip_end_time'] = pd.to_datetime(df['clip_end_time'], format='mixed')
    
    # Sort by Site, labeling_effort, and clip_start_time
    df = df.sort_values(['Site', 'labeling_effort', 'clip_start_time']).reset_index(drop=True)
    
    # Label columns that need OR logic (1 if any row has 1)
    label_cols = ['ECHO', 'BBPC', 'HFPC', 'Whistle', 'Call', 'Boat']
    
    # Columns to keep from first row
    keep_cols = ['labeling_effort', 'Site', 'HydrophoneModel', 'HydrophoneSensitivity', 
                 'original_filename', 'labeled_snippet_filename']
    
    three_sec_windows = []
    
    i = 0
    while i < len(df) - 2:  # Need at least 3 rows
        row1 = df.iloc[i]
        row2 = df.iloc[i + 1]
        row3 = df.iloc[i + 2]
        
        # Check if they're consecutive (1 second apart) and from same site/labeling_effort
        time_diff_1_2 = (row2['clip_start_time'] - row1['clip_start_time']).total_seconds()
        time_diff_2_3 = (row3['clip_start_time'] - row2['clip_start_time']).total_seconds()
        
        same_context = (
            row1['Site'] == row2['Site'] == row3['Site'] and
            row1['labeling_effort'] == row2['labeling_effort'] == row3['labeling_effort']
        )
        
        if same_context and abs(time_diff_1_2 - 1.0) < 0.1 and abs(time_diff_2_3 - 1.0) < 0.1:
            # Create new 3-second window
            new_row = {}
            
            # Clip filename: replace .wav with _s3.wav
            new_row['clip_filename'] = row1['clip_filename'].replace('.wav', '_s3.wav')
            
            # Add individual clip names
            new_row['clip_1'] = row1['clip_filename']
            new_row['clip_2'] = row2['clip_filename']
            new_row['clip_3'] = row3['clip_filename']
            
            # Label columns: OR logic (1 if any has 1)
            for col in label_cols:
                if col in df.columns:
                    # Handle NaN by treating as 0
                    is_nan1 = pd.isna(row1[col])
                    is_nan2 = pd.isna(row2[col])
                    is_nan3 = pd.isna(row3[col])
                    if is_nan1 and is_nan2 and is_nan3:
                        new_row[col] = np.nan
                    else:
                        val1 = 0 if is_nan1 else row1[col]
                        val2 = 0 if is_nan2 else row2[col]
                        val3 = 0 if is_nan3 else row3[col]
                        new_row[col] = 1 if (val1 or val2 or val3) else 0
            
            # Keep columns from first row
            for col in keep_cols:
                if col in df.columns:
                    new_row[col] = row1[col]
            
            # Time columns
            new_row['clip_start_time'] = row1['clip_start_time']
            new_row['clip_end_time'] = row1['clip_start_time'] + timedelta(seconds=3)
            
            three_sec_windows.append(new_row)
            
            # Move to next potential window
            # NOTE: Currently using sliding windows with 1-second stride
            # This means windows overlap (e.g., [0-3s], [1-4s], [2-5s])
            # For non-overlapping windows, change to: i += 3
            i += 1  # Sliding window (1-second stride)
        else:
            # Can't form a 3-second window, move to next row
            i += 1
    
    # Create new DataFrame
    df_3sec = pd.DataFrame(three_sec_windows)
    
    print(f"Original 1-second windows: {len(df)}")
    print(f"Created 3-second windows: {len(df_3sec)}")
    
    return df_3sec

# Create 3-second windows
labels_df_3sec = create_3second_windows(labels_df)
labels_df_3sec.head()

Original 1-second windows: 15518
Created 3-second windows: 9745


,clip_filename,clip_1,clip_2,clip_3,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,Site,HydrophoneModel,HydrophoneSensitivity,original_filename,labeled_snippet_filename,clip_start_time,clip_end_time
0,BSM_20170724_13385800_s3.wav,BSM_20170724_13385800.wav,BSM_20170724_13385900.wav,BSM_20170724_13390000.wav,1,0,0,0,1,1.0,evaluation_v1,BSM,201359382,-172.7,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:58,2017-07-24 13:39:01
1,BSM_20170724_13385900_s3.wav,BSM_20170724_13385900.wav,BSM_20170724_13390000.wav,BSM_20170724_13390100.wav,1,0,0,0,1,1.0,evaluation_v1,BSM,201359382,-172.7,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:38:59,2017-07-24 13:39:02
2,BSM_20170724_13390000_s3.wav,BSM_20170724_13390000.wav,BSM_20170724_13390100.wav,BSM_20170724_13390200.wav,1,0,0,0,1,1.0,evaluation_v1,BSM,201359382,-172.7,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:39:00,2017-07-24 13:39:03
3,BSM_20170724_13390100_s3.wav,BSM_20170724_13390100.wav,BSM_20170724_13390200.wav,BSM_20170724_13390300.wav,1,0,0,0,1,1.0,evaluation_v1,BSM,201359382,-172.7,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:39:01,2017-07-24 13:39:04
4,BSM_20170724_13390200_s3.wav,BSM_20170724_13390200.wav,BSM_20170724_13390300.wav,BSM_20170724_13390400.wav,1,0,0,0,1,1.0,evaluation_v1,BSM,201359382,-172.7,201359382.170724133002.wav,201359382.170724133858.snippet.wav,2017-07-24 13:39:02,2017-07-24 13:39:05


In [18]:
print(labels_df["Boat"].value_counts(dropna=False))
print(labels_df_3sec["Boat"].value_counts(dropna=False))

Boat
NaN    6805
0.0    4358
1.0    4355
Name: count, dtype: int64
Boat
1.0    4231
0.0    3376
NaN    2138
Name: count, dtype: int64


In [19]:
# Check the statistics of the new 3-second windows
print("Label distribution comparison:")
print("\n1-second windows:")
for col in ['ECHO', 'BBPC', 'HFPC', 'Whistle', 'Call', 'Boat']:
    if col in labels_df.columns:
        print(f"{col}: {labels_df[col].sum()} / {len(labels_df)} ({100*labels_df[col].sum()/len(labels_df):.1f}%)")

print("\n3-second windows:")
for col in ['ECHO', 'BBPC', 'HFPC', 'Whistle', 'Call', 'Boat']:
    if col in labels_df_3sec.columns:
        print(f"{col}: {labels_df_3sec[col].sum()} / {len(labels_df_3sec)} ({100*labels_df_3sec[col].sum()/len(labels_df_3sec):.1f}%)")

print("\nSample of 3-second windows:")
labels_df_3sec[['clip_filename', 'clip_1', 'clip_2', 'clip_3', 'ECHO', 'BBPC', 'HFPC', 'Whistle', 'Site', 'clip_start_time', 'clip_end_time']].head(10)

Label distribution comparison:

1-second windows:
ECHO: 5289 / 15518 (34.1%)
BBPC: 984 / 15518 (6.3%)
HFPC: 792 / 15518 (5.1%)
Whistle: 2917 / 15518 (18.8%)
Call: 6660 / 15518 (42.9%)
Boat: 4355.0 / 15518 (28.1%)

3-second windows:
ECHO: 5383 / 9745 (55.2%)
BBPC: 811 / 9745 (8.3%)
HFPC: 747 / 9745 (7.7%)
Whistle: 2793 / 9745 (28.7%)
Call: 6190 / 9745 (63.5%)
Boat: 4231.0 / 9745 (43.4%)

Sample of 3-second windows:


,clip_filename,clip_1,clip_2,clip_3,ECHO,BBPC,HFPC,Whistle,Site,clip_start_time,clip_end_time
0,BSM_20170724_13385800_s3.wav,BSM_20170724_13385800.wav,BSM_20170724_13385900.wav,BSM_20170724_13390000.wav,1,0,0,0,BSM,2017-07-24 13:38:58,2017-07-24 13:39:01
1,BSM_20170724_13385900_s3.wav,BSM_20170724_13385900.wav,BSM_20170724_13390000.wav,BSM_20170724_13390100.wav,1,0,0,0,BSM,2017-07-24 13:38:59,2017-07-24 13:39:02
2,BSM_20170724_13390000_s3.wav,BSM_20170724_13390000.wav,BSM_20170724_13390100.wav,BSM_20170724_13390200.wav,1,0,0,0,BSM,2017-07-24 13:39:00,2017-07-24 13:39:03
3,BSM_20170724_13390100_s3.wav,BSM_20170724_13390100.wav,BSM_20170724_13390200.wav,BSM_20170724_13390300.wav,1,0,0,0,BSM,2017-07-24 13:39:01,2017-07-24 13:39:04
4,BSM_20170724_13390200_s3.wav,BSM_20170724_13390200.wav,BSM_20170724_13390300.wav,BSM_20170724_13390400.wav,1,0,0,0,BSM,2017-07-24 13:39:02,2017-07-24 13:39:05
5,BSM_20170724_13390300_s3.wav,BSM_20170724_13390300.wav,BSM_20170724_13390400.wav,BSM_20170724_13390500.wav,1,0,0,0,BSM,2017-07-24 13:39:03,2017-07-24 13:39:06
6,BSM_20170724_13390400_s3.wav,BSM_20170724_13390400.wav,BSM_20170724_13390500.wav,BSM_20170724_13390600.wav,1,0,0,0,BSM,2017-07-24 13:39:04,2017-07-24 13:39:07
7,BSM_20170724_13390500_s3.wav,BSM_20170724_13390500.wav,BSM_20170724_13390600.wav,BSM_20170724_13390700.wav,1,0,0,0,BSM,2017-07-24 13:39:05,2017-07-24 13:39:08
8,BSM_20170724_13390600_s3.wav,BSM_20170724_13390600.wav,BSM_20170724_13390700.wav,BSM_20170724_13390800.wav,1,0,0,0,BSM,2017-07-24 13:39:06,2017-07-24 13:39:09
9,BSM_20170724_13390700_s3.wav,BSM_20170724_13390700.wav,BSM_20170724_13390800.wav,BSM_20170724_13390900.wav,1,0,0,0,BSM,2017-07-24 13:39:07,2017-07-24 13:39:10


In [20]:
# Optional: Save the 3-second windows DataFrame
output_path = "../data/Verified_Dataset/labels/labels_merged_3sec.csv"
labels_df_3sec.to_csv(output_path, index=False)
print(f"Saved 3-second windows to: {output_path}")

Saved 3-second windows to: ../data/Verified_Dataset/labels/labels_merged_3sec.csv


In [7]:
labels_df[labels_df["labeling_effort"] == "irene_20min"]

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,SnippetFilename,start_s,end_s
261,BSM_20170724_13090979.wav,0,0,1,1,1,0.0,irene_20min,hm,NaN,NaN,hw,it could be classified as bm,NaN,NaN,2017-07-24 13:09:09.790000,NaN,NaN,BSM,NaN,201359382,-172.7,2017-07-24 13:09:09.790000,2017-07-24 13:09:10.790000,NaN,NaN,NaN,NaN,BSM_20170724_13090979.wav,0.0,1.0
262,BSM_20170724_13091079.wav,0,0,0,1,1,0.0,irene_20min,NaN,NaN,NaN,w,NaN,NaN,NaN,2017-07-24 13:09:09.790000,NaN,NaN,BSM,NaN,201359382,-172.7,2017-07-24 13:09:10.790000,2017-07-24 13:09:11.790000,NaN,NaN,NaN,NaN,BSM_20170724_13090979.wav,1.0,2.0
263,BSM_20170724_13091179.wav,0,0,0,1,1,0.0,irene_20min,NaN,NaN,NaN,w,NaN,NaN,NaN,2017-07-24 13:09:09.790000,NaN,NaN,BSM,NaN,201359382,-172.7,2017-07-24 13:09:11.790000,2017-07-24 13:09:12.790000,NaN,NaN,NaN,NaN,BSM_20170724_13090979.wav,2.0,3.0
264,BSM_20170724_13132559.wav,0,0,1,0,1,0.0,irene_20min,NaN,NaN,NaN,h,NaN,NaN,NaN,2017-07-24 13:13:25.590000,NaN,NaN,BSM,NaN,201359382,-172.7,2017-07-24 13:13:25.590000,2017-07-24 13:13:26.590000,NaN,NaN,NaN,NaN,BSM_20170724_13132559.wav,0.0,1.0
265,BSM_20170724_13132659.wav,0,0,1,0,1,0.0,irene_20min,NaN,NaN,NaN,h,"very low SNR for h, I wouldn't consider it for...",NaN,NaN,2017-07-24 13:13:25.590000,NaN,NaN,BSM,NaN,201359382,-172.7,2017-07-24 13:13:26.590000,2017-07-24 13:13:27.590000,NaN,NaN,NaN,NaN,BSM_20170724_13132559.wav,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15486,RDL_20200828_18480725.wav,1,0,1,0,1,1.0,irene_20min,NaN,NaN,NaN,eh,ship,NaN,NaN,2020-08-28 18:48:06.250000,NaN,NaN,RDL,NaN,5675,-176.5,2020-08-28 18:48:07.250000,2020-08-28 18:48:08.250000,NaN,NaN,NaN,NaN,RDL_20200828_18480625.wav,1.0,2.0
15487,RDL_20200828_18480825.wav,1,0,1,0,1,1.0,irene_20min,NaN,NaN,NaN,eh,ship,NaN,NaN,2020-08-28 18:48:06.250000,NaN,NaN,RDL,NaN,5675,-176.5,2020-08-28 18:48:08.250000,2020-08-28 18:48:09.250000,NaN,NaN,NaN,NaN,RDL_20200828_18480625.wav,2.0,3.0
15491,RDL_20200829_13033200.wav,0,1,0,1,1,0.0,irene_20min,cm,NaN,NaN,bw,NaN,NaN,NaN,2020-08-29 13:03:32.000000,NaN,NaN,RDL,NaN,5675,-176.5,2020-08-29 13:03:32.000000,2020-08-29 13:03:33.000000,NaN,NaN,NaN,NaN,RDL_20200829_13033200.wav,0.0,1.0
15492,RDL_20200829_13033300.wav,0,1,0,1,1,0.0,irene_20min,cm,NaN,NaN,bw,NaN,NaN,NaN,2020-08-29 13:03:32.000000,NaN,NaN,RDL,NaN,5675,-176.5,2020-08-29 13:03:33.000000,2020-08-29 13:03:34.000000,NaN,NaN,NaN,NaN,RDL_20200829_13033200.wav,1.0,2.0
